# Learning a PDFA using ALERGIA

This tutorial uses a small synthetic dataset to demonstrate how to:
1. Define a sample of observed sequences;
2. Construct a prefix-tree acceptor;
3. Apply the ALERGIA algorithm to learn a deterministic finite automaton (DFA);
4. Calculate the resulting transition probabilities (PDFA);
5. Inspect the learned PDFA.

## Set up the tutorial

The first step is to import the pdfa_learning package. This can be done using the following code:

In [1]:
from pathlib import Path
import sys

repository_root = Path.cwd().parents[1]

if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import pdfa_learning as pl

## 1. Define a sample of observed sequences

The initial data is provided as a list of sequences. Each string represents one observed sequence, and repeated strings indicate multiple observations of the same sequence. 

We can define a sample of observed sequences as follows:

In [2]:
sequences = [
    "0", "0", "0", "0", "0", "0", "0", "0",
    "01", "01", "01", "01", "01", "01", "01",
    "10", "10", "10", "10", "10",
    "11", "11", "11", "11", "11",
    "12", "12",
    "1", "1", "1",
]

## 2. Construct a prefix-tree acceptor

The alphabet of symbols is implicitly defined by the unique characters present in the sequences.

In [3]:
alphabet = pl.get_alphabet(sequences)
alphabet

['0', '1', '2']

We should next define the initial states that will be used to construct the prefix-tree acceptor. Note that the initial state list contains one identifier for each state in the
prefix-tree acceptor. The artificial starting state is represented by `"*"` and the remaining states are assigned integer identifiers.

In [4]:
states = pl.get_initial_states(sequences)
states

['*', 0, 1, 2, 3, 4, 5, 6]

The next step is to construct a prefix-tree acceptor (PTA) from the observed sequences. 

In this codebase, the PTA is represented as a three-dimensional NumPy array. Its first dimension represents the alphabet symbol, its second dimension represents the current state, and its third dimension represents the destination state.

In [5]:
prefix_tree_acceptor = pl.get_transition_matrix(
    sequences,
    alphabet,
)

The first array index selects the alphabet symbol. Because `"0"` is at position `0` in `alphabet`, the first index is `0`.

The second and third indices select positions in `states`. State `0` is at position `1`, while state `1` is at position `2`. Therefore, the following retrieves the number of transitions from state `0` to state `1` using symbol `"0"`.

In [6]:
prefix_tree_acceptor[0, 1, 2]

15

The full matrix representation of the prefix-tree acceptor can be printed as follows:

In [7]:
prefix_tree_acceptor

array([[[ 0, 30,  0,  0,  0,  0,  0,  0],
        [ 0,  0, 15,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  5,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0]],

       [[ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0, 15,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  7,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  5,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0]],

       [[ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  2],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  

## 3. Apply the ALERGIA algorithm to learn a deterministic finite automaton (DFA)

The next step is to actually apply the ALERGIA algorithm to the prefix-tree acceptor. 

ALERGIA merges statistically compatible states in the prefix-tree acceptor. The resulting array contains the transition counts associated with the learned automaton. These counts are converted into transition probabilities in the following section.

Before we do this though, it is worth briefly describing some of the parameters of the `alergia` function:
- `alpha`: This is the significance level used in the statistical test for merging states. Smaller values generally permit more merges, while larger values generally permit fewer merges. Must lie in the interval ``(0, 2]``.
- `method : {"carrasco", "de_la_higuera"}`: Method used to test whether two states are compatible during the state-merging process. ``"carrasco"`` follows the compatibility test presented by Carrasco and Oncina [1], while ``"de_la_higuera"`` follows the red-blue merge formulation presented by de la Higuera [2]. 
- `output_level : {"Suppressed", "Truncated", "Full"}`: Controls the amount of progress information printed during the state-merging process

In [8]:
learned_dfa, learned_states, tracking = pl.alergia(
    prefix_tree_acceptor,
    states,
    alphabet,
    alpha=0.2,
    method="carrasco",
)

Let's inspect the output of the `alergia` function. The first output is the learned DFA, which is represented in the same way as the PTA as shown below:

In [9]:
learned_dfa

array([[[ 0, 30,  0,  0],
        [ 0,  0, 15,  0],
        [ 0,  0,  5,  0],
        [ 0,  0,  0,  0]],

       [[ 0,  0,  0,  0],
        [ 0,  0, 15,  0],
        [ 0,  0,  0, 12],
        [ 0,  0,  0,  0]],

       [[ 0,  0,  0,  0],
        [ 0,  0,  0,  0],
        [ 0,  0,  2,  0],
        [ 0,  0,  0,  0]]])

The learned states are also returned as a list, which can be printed as follows:

In [10]:
learned_states

['*', 0, 1, 3]

`tracking` is a dictionary that contains information about the state-merging process, which can be used for debugging or analysis purposes. You can print this dictionary as follows:

In [11]:
tracking

{'initial_states': 8,
 'final_states': 4,
 'attempted_merges': 16,
 'successful_merges': 3,
 'recursive_merge_attempts': 5,
 'recursive_merge_failures': 2}

### 3.1 Choosing the alpha Parameter

The `alpha` parameter controls the tolerance used when deciding whether two states are statistically compatible and may therefore be merged. 

In this implementation: 
- smaller `alpha` values produce a wider compatibility bound and generally allow more state merges; 
- larger `alpha` values produce a narrower compatibility bound and generally result in fewer state merges. 

Consequently, smaller values tend to produce more compact, generalised automata, whereas larger values tend to preserve more of the structure present in the original prefix tree.

The appropriate value of alpha is application dependent and may be selected by comparing the resulting automata using validation data or other model selection criteria. The evaluation functions provided by the package can be used to compare candidate values using validation data.

### 3.2 Controlling Print Output

The amount of information printed while the algorithm runs can be controlled using the `output_level` parameter:
- `"Suppressed"` prints no progress information and is the default. 
- `"Truncated"` prints the main state comparisons and merge outcomes. 
- `"Full"` additionally prints iteration numbers and details of recursive merges.

## 4. Calculate the resulting transition probabilities (PDFA)

We can now convert the learned DFA into a PDFA by calculating the transition probabilities:

In [25]:
pdfa = pl.probability_transition_matrix(learned_dfa, learned_states, alphabet)
pdfa

array([[[0.        , 1.        , 0.        , 0.        ],
        [0.        , 0.        , 0.5       , 0.        ],
        [0.        , 0.        , 0.13513514, 0.        ],
        [0.        , 0.        , 0.        , 0.        ]],

       [[0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.5       , 0.        ],
        [0.        , 0.        , 0.        , 0.32432432],
        [0.        , 0.        , 0.        , 0.        ]],

       [[0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.05405405, 0.        ],
        [0.        , 0.        , 0.        , 0.        ]]])

For a given state, the outgoing transition probabilities may sum to less than one. The remaining probability mass represents the probability that a sequence terminates at that state.

Note this function can also be used to convert a PTA into its probabilistic form as well, if desired.

## 5. Inspect the learned PDFA

The probability matrix can then be used with the sequence and pattern probability functions provided by the `pdfa_learning` package.

For example, if we wished to calculate the probability assigned to the sequence `"01"` by the learned PDFA, we could do so as follows:

In [26]:
pl.probability_estimate_of_exact_sequence(
    p_mat=pdfa,
    sequence="01",
    alphabet=alphabet,
)

0.16216216216216217

We can also calculate the probabilities of patterns occurring in the learned PDFA. A pattern differs from a sequence in that the symbols occur in the specified order, but not necessarily consecutively.

For example, if we wished to calculate the the probability assigned to the pattern `"01"` by the learned PDFA, we could do so as follows:

In [32]:
probability_of_pattern = pl.probability_estimate_of_pattern(
    pdfa,
    pattern="01",
    alphabet=alphabet,
)
probability_of_pattern

array([0.7       , 0.22857143, 0.05714286, 0.        ])

We can infer the following about the probability of the pattern `01` from the result:

In [34]:
for state, probability in zip(learned_states, probability_of_pattern):
    print(
        f"Probability of observing the pattern starting from "
        f"state {state!r}: {probability:.3f}"
    )

Probability of observing the pattern starting from state '*': 0.700
Probability of observing the pattern starting from state 0: 0.229
Probability of observing the pattern starting from state 1: 0.057
Probability of observing the pattern starting from state 3: 0.000


## Summary

In this tutorial, we constructed a prefix-tree representation of the observed sequences, applied ALERGIA to merge statistically compatible states, and converted the learned transition counts into probabilities.

The next tutorial visualises the resulting automaton.

## References

[1] Carrasco, R. C. and Oncina, J. (1994). "Learning Stochastic 
Regular Grammars by Means of a State Merging Method." In *Grammatical 
Inference and Applications*, pp. 139–152. 

[2] de la Higuera, C. (2010). *Grammatical Inference: Learning 
Automata and Grammars*. Cambridge University Press.